# Monthly GFS T2M with the grib2io xarray backend

This notebook builds a **month-long time series of GFS 2-metre temperature (T2M)** from
NOAA's public S3 bucket (`noaa-gfs-bdp-pds`) without downloading the files.

## How it stays fast

Two features combine to make per-file manifest generation near-instant:

1. **`.idx` sidecar** — grib2io automatically fetches the small `url + ".idx"` text file
   that NOAA publishes alongside every GFS file. This gives message byte offsets
   directly, so grib2io only reads the compact header bytes (Sections 0–5) rather
   than streaming the entire ~500 MB GRIB2 object.

2. **Variable filters** — passing `filters={"shortName": "TMP", ...}` to
   `xr.open_dataset(..., engine="grib2io", use_icechunk=True)` or
   `xr.open_mfdataset(..., engine="grib2io", use_icechunk=True)` means only
   the single T2M message per file is indexed; all other variables are skipped.
   A full GFS 0.25° file contains ~700 variables; filtering to one reduces
   manifest work by ~700×.

Notebook convention: pass grib2io-specific options directly as keyword arguments
to `xr.open_dataset(...)` / `xr.open_mfdataset(...)`.

For a full-file example (all variables, VirtualiZarr integration) see
[grib2io_s3_virtualizarr.ipynb](grib2io_s3_virtualizarr.ipynb).

**Dependencies:** `grib2io[icechunk]`, `s3fs`, `matplotlib`


## Notebook Convention

Use the standard grib2io xarray backend interface directly.
Pass grib2io-specific options as keyword arguments to `xr.open_dataset(...)` or `xr.open_mfdataset(...)` (for example `use_icechunk`, `storage_options`, `filters`, `max_workers`, `network_timeout`, and `max_concurrent_requests`).

In [ ]:
import time
import warnings

from dask import config as dask_config
import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr

warnings.filterwarnings("ignore")

# Optional distributed scheduler for faster remote-read task throughput.
USE_DISTRIBUTED = True
if USE_DISTRIBUTED:
    try:
        from dask.distributed import Client, LocalCluster

        cluster = LocalCluster(
            n_workers=4,
            threads_per_worker=1,
            processes=True,
            dashboard_address=":8787",
        )
        client = Client(cluster)
        print(client)
    except Exception as exc:
        print(f"distributed not available, using default scheduler: {type(exc).__name__}: {exc}")
        client = None
else:
    client = None

# Small scheduler tuning for reduction-heavy graphs.
dask_config.set({"array.slicing.split_large_chunks": False})


def compute(arr, *, max_attempts=5, base_sleep=2):
    """compute() with automatic retry on transient network errors."""
    for attempt in range(1, max_attempts + 1):
        try:
            return arr.compute()
        except Exception as exc:
            msg = str(exc).lower()
            transient = (
                "timeout" in msg
                or "timed out" in msg
                or "connect" in msg
                or "dns error" in msg
                or "nodename nor servname" in msg
                or "icechunkerror" in type(exc).__name__.lower()
            )
            if (not transient) or attempt == max_attempts:
                raise
            sleep_s = base_sleep**attempt
            print(f"Transient error on attempt {attempt}/{max_attempts}, retrying in {sleep_s}s: {exc}")
            time.sleep(sleep_s)


# Optional notebook-only dependency for future VirtualiZarr/object-store workflows.
try:
    from obstore.store import S3Store
    from obspec_utils.registry import ObjectStoreRegistry

    OBSTORE_AVAILABLE = True
    print("Optional obstore dependencies are available.")
except ImportError:
    OBSTORE_AVAILABLE = False

## 1. Configuration

Set the month, grid, and the GRIB2 message filter that selects 2-metre temperature.
`typeOfFirstFixedSurface = 103` is the GRIB2 Table 4.5 code for
*"Specified height level above ground (m)"*, and `level = 2` selects the 2 m level.


In [ ]:
BUCKET = "noaa-gfs-bdp-pds"
YEAR = "2025"  # any YYYY present in the bucket
GRID = "0p25"  # 0.25-degree global grid
CYCLE = "00"  # 00Z analysis
FORECAST = "f000"  # analysis hour (zero-lead)

# Keep Icechunk enabled for fast virtualized reads.
USE_ICECHUNK = True

# More tolerant network settings for remote NOAA S3 reads via s3fs/botocore.
storage_options = {
    "anon": True,
    "config_kwargs": {
        "connect_timeout": 30,
        "read_timeout": 120,
        "retries": {"max_attempts": 10, "mode": "adaptive"},
    },
}

# Filter to 2-metre temperature only — skips all other variables during scanning.
# typeOfFirstFixedSurface=103 -> "height above ground (m)"; level=2 -> 2 m
T2M_FILTERS = {"shortName": "TMP", "typeOfFirstFixedSurface": 103}

# Build URLs for the selected year.
year_start = pd.to_datetime(f"{YEAR}-01-01")
year_end = pd.to_datetime(f"{YEAR}-12-31")
dates = pd.date_range(year_start, year_end, freq="D")
PERIOD_LABEL = YEAR
urls = [f"s3://{BUCKET}/gfs.{d.strftime('%Y%m%d')}/{CYCLE}/atmos/gfs.t{CYCLE}z.pgrb2.{GRID}.{FORECAST}" for d in dates]
urls2 = [f"s3://{BUCKET}/gfs.{d.strftime('%Y%m%d')}/{CYCLE}/atmos/gfs.t{CYCLE}z.pgrb2.{GRID}.f120" for d in dates]
print(f"Period: {PERIOD_LABEL}  ({len(urls)} files)")
print("First:", urls[0])
print("Last: ", urls[-1])
print(f"use_icechunk={USE_ICECHUNK}")

## 2. Verify: single file opens quickly

With the `.idx` sidecar + T2M filter, manifest generation should take only a
few seconds even for a 500 MB remote file.


In [ ]:
import time

t0 = time.perf_counter()
ds_sample = xr.open_dataset(
    urls[0],
    engine="grib2io",
    use_icechunk=USE_ICECHUNK,
    storage_options=storage_options,
    filters=T2M_FILTERS,
    max_workers=2,
    network_timeout=300,
    max_concurrent_requests=2,
    chunks={},
)
print(f"Opened in {time.perf_counter() - t0:.1f}s")
print(ds_sample)

In [ ]:
# With the filter applied there is exactly one TMP variable (no pressure-level
# TMP in this dataset), so we can reference it by name directly.
T2M_VAR = "TMP"
print(ds_sample[T2M_VAR])

## 3. Build the monthly T2M dataset

Loop over each day with the T2M filter applied.  Only the tiny `.idx` sidecar
and the matching message headers are fetched from S3 — no data payloads yet.
Temperature values are pulled on demand when `.compute()` is called later.


In [ ]:
import time

t0 = time.perf_counter()
# Lower worker/concurrency counts reduce bursty S3 traffic and timeout risk.
ds = xr.open_mfdataset(
    urls,
    engine="grib2io",
    use_icechunk=USE_ICECHUNK,
    storage_options=storage_options,
    filters=T2M_FILTERS,
    max_workers=4,
    network_timeout=300,
    max_concurrent_requests=2,
    chunks={},
)

elapsed = time.perf_counter() - t0
print(f"Scanned {len(urls)} files in {elapsed:.1f}s  ({elapsed / len(urls):.2f}s/file)")
print(ds)
ds120 = xr.open_mfdataset(
    urls2,
    engine="grib2io",
    use_icechunk=USE_ICECHUNK,
    storage_options=storage_options,
    filters=T2M_FILTERS,
    max_workers=4,
    network_timeout=300,
    max_concurrent_requests=2,
    chunks={},
)

In [ ]:
# The multi-file xarray open produced a dataset with a valid_time dimension.
# Select TMP and squeeze height_above_ground if present.
t2m_monthly = ds[T2M_VAR]
if "height_above_ground" in t2m_monthly.dims:
    t2m_monthly = t2m_monthly.isel(height_above_ground=0)
t2m120 = ds120[T2M_VAR].isel(height_above_ground=0)
# Convert K -> degC for readability
t2m_monthly = t2m_monthly - 273.15
t2m120 = t2m120 - 273.15

t2m_monthly.attrs["units"] = "degC"
t2m120.attrs["units"] = "degC"
t2m_monthly.attrs["long_name"] = "2-metre Temperature"
t2m120.attrs["long_name"] = "2-metre Temperature"
print(t2m_monthly)
print(t2m120)

## 4. Global map — mean T2M for the month

`.mean("valid_time")` is computed first (still lazy), then `.compute()` fetches
only the bytes needed to materialise those values.

In [ ]:
print("Computing monthly mean ...")

t2m_mean = compute(t2m_monthly.mean("valid_time"))

# ---- plot ----------------------------------------------------------------
lats = t2m_mean.coords.get("latitude", t2m_mean.coords.get("y", None))
lons = t2m_mean.coords.get("longitude", t2m_mean.coords.get("x", None))

fig, ax = plt.subplots(figsize=(14, 6))
if lats is not None and lons is not None:
    pcm = ax.pcolormesh(lons, lats, t2m_mean.values, cmap="RdBu_r", shading="auto")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
else:
    pcm = ax.imshow(t2m_mean.values, origin="upper", cmap="RdBu_r", aspect="auto")
    ax.set_xlabel("x index")
    ax.set_ylabel("y index")

plt.colorbar(pcm, ax=ax, label="degC", shrink=0.7)
ax.set_title(f"GFS 00Z Analysis - Mean 2-m Temperature\n{PERIOD_LABEL}  (00Z, {GRID}deg)")
plt.tight_layout()
plt.show()

## 5. Point time series

Select the nearest grid point to a location of interest and plot the daily
T2M over the month.  Only the bytes for those specific grid-point chunks
are fetched.

In [ ]:
# Location of interest (change as desired)
TARGET_LAT = 40.71  # New York City
TARGET_LON = -74.01  # (negative = West)

lats = t2m_monthly.coords.get("latitude", t2m_monthly.coords.get("y", None))
lons = t2m_monthly.coords.get("longitude", t2m_monthly.coords.get("x", None))

# Compute 1-D global-mean time series with moderate time chunking.
ts = compute(t2m_monthly.chunk({"valid_time": 14}).mean("x").mean("y"))
ts120 = compute(t2m120.chunk({"valid_time": 14}).mean("x").mean("y"))
times = pd.to_datetime(ts.coords["valid_time"].values)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(times, ts.values.ravel(), marker="o", linewidth=1.5, markersize=4, label="analysis")
ax.plot(times, ts120.values.ravel(), marker="*", color="green", linewidth=1.5, markersize=4, label="120h forecast")
ax.axhline(float(ts.mean()), color="red", linestyle="--", linewidth=1, label="analysis yearly mean")
ax.axhline(float(ts120.mean()), color="green", linestyle="--", linewidth=1, label="120h forecast yearly mean")
ax.set_xlabel("Date")
ax.set_ylabel("T2M (\u00b0C)")
ax.set_title(f"GFS 00Z 2-m Temperature \u2014 {PERIOD_LABEL} Global Average")
ax.legend()
ax.grid(alpha=0.4)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(f"\nYearly mean: {float(ts.mean()):.2f} \u00b0C")
print(f"Min: {float(ts.min()):.2f} \u00b0C  on {times[int(ts.argmin())].date()}")
print(f"Max: {float(ts.max()):.2f} \u00b0C  on {times[int(ts.argmax())].date()}")

## 6. Spatial standard deviation — day-to-day variability

Where is month-to-month variability highest?  Compute standard deviation
along `valid_time` and plot.

In [ ]:
print("Computing temporal standard deviation ...")

# Smaller time chunks reduce pressure when reducing over a full year.
t2m_std = compute(t2m_monthly.chunk({"valid_time": 7}).std("valid_time"))
t2m120_std = compute(t2m120.chunk({"valid_time": 7}).std("valid_time"))
lats = t2m_std.coords.get("latitude", t2m_std.coords.get("y", None))
lons = t2m_std.coords.get("longitude", t2m_std.coords.get("x", None))

fig, ax = plt.subplots(figsize=(14, 6))
if lats is not None and lons is not None:
    pcm = ax.pcolormesh(lons, lats, t2m_std - t2m120_std, cmap="YlOrRd", shading="auto")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
else:
    pcm = ax.imshow(t2m_std - t2m120_std, origin="upper", cmap="YlOrRd", aspect="auto")

plt.colorbar(pcm, ax=ax, label="\u00b0C", shrink=0.7)
ax.set_title(f"GFS 00Z \u2014 Day-to-Day T2M Standard Deviation\n{PERIOD_LABEL}  ({GRID}\u00b0)")
plt.tight_layout()
plt.show()